In [ ]:
# Use termcolor to make it easy to colorize the outputs.
!pip install termcolor > /dev/null
!pip install langchain
!pip install openai
!pip install langchain_experimental
!pip install tiktoken
!pip install faiss-cpu==1.7.4


In [ ]:
!pip install FAISS-cpu
!pip install langchain
!pip install langchain-core
!pip install langchain-openai
!pip install langchain.chat_models
!pip install langchain.docstore
!pip install langchain.embeddings
!pip install langchain.retrievers
!pip install langchain.vectorstores

In [ ]:
!pip install langchain-classic

In [ ]:
from datetime import datetime, timedelta
from typing import List
import math
import faiss
import os
import logging
logging.basicConfig(level=logging.ERROR)
from langchain_openai import ChatOpenAI
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_classic.retrievers import TimeWeightedVectorStoreRetriever
from langchain_community.vectorstores import FAISS
from termcolor import colored
from langchain_experimental.generative_agents import (

    GenerativeAgent,
    GenerativeAgentMemory,
)

In [5]:
import os
from google.colab import userdata

# Retrieve the secret key from Colab
openai_key = userdata.get('OPENAI_API_KEY')

# Set it as an environment variable for OpenAI/LangChain libraries
os.environ["OPENAI_API_KEY"] = openai_key

print("OpenAI API key loaded successfully!")

OpenAI API key loaded successfully!


In [6]:
USER_NAME = "Nayan"  # The name you want to use when interviewing the agent.

LLM = ChatOpenAI(max_tokens=1500)  # Can be any LLM you want.

## Implementing Your First Generative Agent




In [7]:


def relevance_score_fn(score: float) -> float:
    """Return a similarity score on a scale [0, 1]."""
    # This will differ depending on a few things:
    # - the distance / similarity metric used by the VectorStore
    # - the scale of your embeddings (OpenAI's are unit norm. Many others are not!)
    # This function converts the euclidean norm of normalized embeddings
    # (0 is most similar, sqrt(2) most dissimilar)
    # to a similarity function (0 to 1)
    return 1.0 - score / math.sqrt(2)


def create_new_memory_retriever():
    """Create a new vector store retriever unique to the agent."""
    # Define your embedding model
    embeddings_model = OpenAIEmbeddings()
    # Initialize the vectorstore as empty
    embedding_size = 1536
    index = faiss.IndexFlatL2(embedding_size)
    vectorstore = FAISS(
        embeddings_model.embed_query,
        index,
        InMemoryDocstore({}),
        {},
        relevance_score_fn=relevance_score_fn,
    )
    return TimeWeightedVectorStoreRetriever(
        vectorstore=vectorstore, other_score_keys=["importance"], k=15
    )

In [8]:
alexis_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=8,  # we will give this a relatively low number to show how reflection works
)

# Defining the Generative Agent: Alexis
alexis = GenerativeAgent(
    name="Alexis",
    age=30,
    traits="curious, creative writer, world traveler",  # Persistent traits of Alexis
    status="exploring the intersection of technology and storytelling",  # Current status of Alexis
    memory_retriever=create_new_memory_retriever(),
    llm=LLM,
    memory=alexis_memory,
)

In [9]:
# The current "Summary" of a character can't be made because the agent hasn't made
# any observations yet.
print(alexis.get_summary())

Name: Alexis (age: 30)
Innate traits: curious, creative writer, world traveler
Alexis is direct and honest, with a strong sense of responsibility and a tendency to take things seriously. She is diligent and has a practical approach to solving problems.


In [ ]:
# We can add memories directly to the memory object

alexis_observations = [
    "Alexis recalls her morning walk in the park",
    "Alexis feels excited about the new book she started reading",
    "Alexis remembers her conversation with a close friend",
    "Alexis thinks about the painting she saw at the art gallery",
    "Alexis is planning to learn a new recipe for dinner",
    "Alexis is looking forward to her weekend trip",
    "Alexis contemplates her goals for the month."
]

for observation in alexis_observations:
    alexis.memory.add_memory(observation)



# We will see how this summary updates after more observations to create a more rich description.
print(alexis.get_summary(force_refresh=True))

## Interacting and Providing Context to Generative Characters

## Pre-Interview with Character

Before sending our character on their way, let's ask them a few questions.

In [11]:
def interview_agent(agent: GenerativeAgent, message: str) -> str:
    """Help the notebook user interact with the agent."""
    new_message = f"{USER_NAME} says {message}"
    return agent.generate_dialogue_response(new_message)[1]

In [12]:
interview_agent(alexis, "What do you like to do?")


'Alexis said "I enjoy writing, traveling, and exploring different cultures. How about you, Nayan? What do you like to do?"'

## Step through the day's observations.

In [13]:
# Let's give Alexa a series of observations to reflect on her day
# Adding observations to Alexis' memory
alexis_observations_day = [
    "Alexis starts her day with a refreshing yoga session.",
    "Alexis spends time writing in her journal.",
    "Alexis experiments with a new recipe she found online.",
    "Alexis gets lost in her thoughts while gardening.",
    "Alexis decides to call her grandmother for a heartfelt chat.",
    "Alexis relaxes in the evening by playing her favorite piano pieces.",
]

for observation in alexis_observations_day:
    alexis.memory.add_memory(observation)


In [14]:
# Let's observe how Alexis's day influences her memory and character
for i, observation in enumerate(alexis_observations_day):
    _, reaction = alexis.generate_reaction(observation)
    print(colored(observation, "green"), reaction)
    if ((i + 1) % len(alexis_observations_day)) == 0:
        print("*" * 40)
        print(
            colored(
                f"After these observations, Alexis's summary is:\n{alexis.get_summary(force_refresh=True)}",
                "blue",
            )
        )
        print("*" * 40)


Alexis starts her day with a refreshing yoga session. Alexis smiles, feeling energized and ready to tackle the day ahead.
Alexis spends time writing in her journal. Alexis smiles, feeling productive and reflective as she writes in her journal.
Alexis experiments with a new recipe she found online. Alexis feels excited to try out the new recipe and adds it to her meal plan for the week.
Alexis gets lost in her thoughts while gardening. Alexis takes a moment to appreciate the peace and tranquility of nature while gardening.
Alexis decides to call her grandmother for a heartfelt chat. Alexis said "Hi Grandma, it's Alexis. I just wanted to hear your voice and tell you how much I love and appreciate you."
Alexis relaxes in the evening by playing her favorite piano pieces. Alexis smiles, feeling relaxed and content while playing her favorite piano pieces.
****************************************
After these observations, Alexis's summary is:
Name: Alexis (age: 30)
Innate traits: curious, cre

## Adding Multiple Characters



In [15]:
# Creating Jordan's Memory
jordan_memory = GenerativeAgentMemory(
    llm=LLM,
    memory_retriever=create_new_memory_retriever(),
    verbose=False,
    reflection_threshold=7,  # Set to illustrate Jordan's reflective capabilities
)

# Defining the Generative Agent: Jordan
jordan = GenerativeAgent(
    name="Jordan",
    age=28,
    traits="tech enthusiast, avid gamer, foodie",  # Persistent traits of Jordan
    status="navigating the world of tech startups",  # Current status of Jordan
    memory_retriever=create_new_memory_retriever(),
    llm=LLM,
    memory=jordan_memory,
)

# Adding observations to Jordan's memory
jordan_observations_day = [
    "Jordan finished a challenging coding project last night",
    "Jordan won a local gaming tournament over the weekend",
    "Jordan tried a new sushi restaurant and loved it",
    "Jordan read an article about the latest AI advancements",
    "Jordan is planning a meetup with tech enthusiasts",
    "Jordan discovered a bug in his latest app prototype",
    "Jordan booked tickets for a tech conference next month",
    "Jordan feels excited about a potential startup idea",
    "Jordan spent the evening playing video games to unwind",
    "Jordan is considering enrolling in a machine learning course"
]

for observation in jordan_observations_day:
    jordan.memory.add_memory(observation)

print(jordan.get_summary())

Name: Jordan (age: 28)
Innate traits: tech enthusiast, avid gamer, foodie
Jordan is a tech-savvy individual who is passionate about gaming, coding, startups, machine learning, and AI advancements. He enjoys connecting with like-minded individuals at tech meetups and conferences. Additionally, he values relaxation and trying new experiences, as seen through his interest in sushi and video games.


## Dialogue between Generative Agents



In [16]:
def run_conversation(agents: List[GenerativeAgent], initial_observation: str) -> None:
    """Runs a conversation between agents."""
    _, observation = agents[1].generate_reaction(initial_observation)
    print(observation)
    max_turns = 3
    turns = 0
    while turns<=max_turns:
        break_dialogue = False
        for agent in agents:
            stay_in_dialogue, observation = agent.generate_dialogue_response(
                observation
            )
            print(observation)
            # observation = f"{agent.name} said {reaction}"
            if not stay_in_dialogue:
                break_dialogue = True
        if break_dialogue:
            break
        turns += 1

In [17]:
agents = [alexis, jordan]
run_conversation(
    agents,
    "Alexis said: Hey Jordan, I've been exploring how technology influences creativity lately. Since you're into tech, I was wondering if you've seen any interesting intersections in your field?",
)




Jordan said "That's an interesting topic! I've actually noticed some cool intersections between tech and creativity, especially in the world of AI and design thinking."
Alexis said "That's fascinating! I've been exploring how technology can enhance storytelling and creativity in my own work. It's amazing to see the possibilities that emerge when these two worlds intersect."
Jordan said "That's really cool, Alexis! I totally agree, the fusion of technology and creativity can lead to some incredible innovations. Have you come across any specific examples or projects that have inspired you in this realm?"
Alexis said "I'm glad you think so, Jordan! One project that has really inspired me is the use of VR technology in creating immersive storytelling experiences. The possibilities are endless! Have you explored any specific examples in this area?"
Jordan said "That's awesome, Alexis! VR technology definitely has the potential to revolutionize storytelling. One project that caught my attent

KeyboardInterrupt: 

## Let's interview our agents after their conversation

Since the generative agents retain their memories from the day, we can ask them about their plans, conversations, and other memoreis.

In [18]:
interview_agent(jordan, "How was your conversation with Alexis?")

'Jordan said "It was great! We had some really interesting discussions about the intersection of tech and creativity. Alexis is always full of great insights. How about you, have you been exploring any new tech trends lately?"'

In [19]:
interview_agent(alexis, "How was your conversation with Jordan?")

'Alexis said "My conversation with Jordan was really inspiring! We talked about the intersection of technology and storytelling, and I learned about some fascinating new projects in that realm. It\'s amazing how technology is shaping the creative landscape. How about you, Nayan? Have you had any interesting conversations lately?"'